<div class="frontmatter text-center">
<h1>Geospatial Data Science</h1>
<h2>Lecture 4: Big Spatial Data</h2>
<h3>IT University of Copenhagen, Spring 2026</h3>
<h3>Instructor: Michael Szell</h3>
</div>

This notebook was created by Ane Rahbek Vierø, adapted from:
* [Tutorial on efficient GeoPandas](https://github.com/martinfleis/efficient-geopandas-workshop)
* [Tutorial on spatial indexes](https://github.com/gboeing/ppd599/blob/2017/19-Spatial-Analysis-and-Cartography/rtree-spatial-indexing.ipynb)
* [User Guide to Dask-GeoPandas](https://dask-geopandas.readthedocs.io/en/stable/guide.html)
* [Intro to Dask-GeoPandas](https://dask-geopandas.readthedocs.io/en/stable/guide/basic-intro.html)
* [Tutorial on H3](https://github.com/uber/h3-py-notebooks/blob/master/notebooks/unified_data_layers.ipynb)
* [Spatial Thoughts' Tutorial on H3](https://spatialthoughts.com/2020/07/01/point-in-polygon-h3-geopandas/)

# Working with large geospatial data sets

In this notebook we will learn:
- how to utilize spatial indexes in GeoPandas
- tips and tricks for using GeoPandas efficiently
- how to work with Dask-GeoPandas
- using H3 to index and analyze data

## Spatial Indexes with GeoPandas

You can see the documentation for the spatial index (`sindex`) in GeoPandas [here](https://geopandas.org/en/stable/docs/reference/sindex.html).

In [ ]:
import geopandas as gpd
from tqdm import tqdm
import contextily as cx

**First, let's load two geodataframes with a polygon and a point datasets:**

The polygons are the voting areas of Denmark. They can be regarded as the smallest administrative areas of the country.

The point dataset represents all the trees in Denmark and come from [GeoDanmark](https://datafordeler.dk/dataoversigt/?emne=landkort%20og%20geografi).

The data were originally downloaded in a [GML](https://en.wikipedia.org/wiki/Geography_Markup_Language) format, having over 2.3 GB (zipped 94 MB).
Reading the dataset into a GeoDataframe from GML can be very slow (~3 minutes on a mac with 16gb RAM).

In [ ]:
%%time
# NOTE Only run this cell if you want to confirm that this indeed is slow
# The data set is not provided here

# trees = gpd.read_file("files/trees.gml")

Instead, we can use the much faster file format [GeoParquet](https://geoparquet.org/) based on Apache Parquet (~10 seconds on the same laptop). Here we have stripped all data from the trees data set except for `objectid` and `geometry`.

In [ ]:
%%time
trees = gpd.read_parquet('files/trees.parquet')
print("Shape of data set: " + str(trees.shape))
trees.head()

In [ ]:
# The tree dataset is quite large, so we only plot a subset
ax = trees.sample(100_000).plot(color="green", markersize=1)
ax.set_title("Trees")
cx.add_basemap(ax=ax, crs=trees.crs, source=cx.providers.CartoDB.Voyager)
ax.set_axis_off()

Load voting districts:

In [ ]:
areas = gpd.read_file("files/voting_areas.gpkg")
print(areas.shape)
areas.head()

In [ ]:
ax = areas.plot(color="purple", edgecolor='white', linewidth=0.2)
ax.set_title("Area Polygons")
cx.add_basemap(ax=ax, crs=areas.crs, source=cx.providers.CartoDB.Voyager)
ax.set_axis_off()

Restrict to Copenhagen

In [ ]:
areas[areas.municipal_id=='0101']

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(20,20))
ax.set_axis_off()
areas[areas.municipal_id=='0101'].plot(ax=ax,categorical=True, column='area_name',cmap='Blues',legend=True)

GeoPandas also supports the [feather](https://geopandas.org/en/latest/docs/user_guide/io.html#apache-parquet-and-feather-file-formats) file format:

In [ ]:
trees_feather = gpd.read_feather('files/trees.feather')
print("Shape of data set: " + str(trees_feather.shape))

In [ ]:
del trees_feather

Let's first check if our data have a spatial index:

In [ ]:
areas.sindex

We can see here that is is an *R Tree Index* using [PyGEOS](https://pygeos.readthedocs.io/en/stable/strtree.html).

In [ ]:
areas.sindex.is_empty

In [ ]:
trees.sindex

In [ ]:
trees.sindex.is_empty

Never versions of GeoPandas automatically creates a spatial index. For other libraries and programming languages, you might need to create it manually.

### Spatial intersection with spatial indexes

To illustrate the difference between using a spatial index and not, let's try to iterate* through the areas dataset and find for each polygon find the trees that intersect with it.

In the first version we first use a simple intersect query on the tree and area geometries:

**Using a for-loop is of course not an efficient way of writing code in general, but it is good for illustrating performance benefits:*

For this demonstration, we will just use a subset of the trees dataset:

In [ ]:
trees_subset = trees[0:200000]

In [ ]:
matchesperarea = []
for i, area in tqdm(areas.iterrows(), total=len(areas)):
    # spatial_index.query() for every area
    possible_matches_index = list(trees_subset.sindex.query(area.geometry))
    possible_matches = trees_subset.iloc[possible_matches_index]
    # run true intersection query only on possible matches
    actual_matches = possible_matches[possible_matches.intersects(area.geometry)]
    matchesperarea.append(len(actual_matches))

In [ ]:
plt.hist(matchesperarea, range(min(matchesperarea), max(matchesperarea),100), log=True)
plt.title("Frequency of matching trees per area");

We *explicitly* used the spatial index of the trees dataset to:
* first, find possible matches between the location of trees and our area data set.
* second, find exact intersections among the possible matches.

This is one way of utilizing spatial indexes by first creating approximate results. Using this method we only have to check for the exact intersections for points whose bounding box intersects with the polygon geometry.

In this case, forcing GeoPandas to use the spatial index offers modest performance improvements, but depending on the shape of the data it can be several times faster.

The fastest way of finding the intersections between our trees and areas is however using GeoPandas build in method `sjoin`:

In [ ]:
%%time

joined_trees_inter = gpd.sjoin(
        trees,
        areas[['ValgstedId', 'geometry']],
        how="left", # options are left, right, inner or outer
        predicate="intersects", # can be contains, crosses, overlaps, within, etc.
    )

joined_trees_inter.drop("index_right", axis=1, inplace=True)

Using `sjoin` with the parameters `left` and `intersects` returns for each tree the id of the intersecting area. (Note that this potentially is a one-to-many join where there could be more than one match for each tree, if we had overlapping polygons.)

Using `sjoin` is quite fast since it automatically uses a spatial index for the data, but in this case it can be even faster if we tweak some of the settings:

In [ ]:
%%time

joined_trees_within = gpd.sjoin(
        trees,
        areas[['ValgstedId', 'geometry']],
        how="left",
        predicate="within", # Changing intersects to within
)

joined_trees_within.drop("index_right", axis=1, inplace=True)

Using `within` rather than `intersects` is technically a bit different, but for point datasets it wil usually give the same results. Using `within` however makes GeoPandas use the spatial index of the trees dataset rather than the spatial index of the areas dataset. Since the point dataset with trees is *much* larger than our polygon dataset with areas, we want to use the spatial index of the larger dataset.

The same happens based on whether we call:

In [ ]:
%%time

areas[['geometry','ValgstedId']].sjoin(trees, predicate="intersects", how="right")

**or:**

In [ ]:
%%time

trees.sjoin(areas[['geometry','ValgstedId']], predicate="intersects", how="left")

The two joins are the same and give the same result - switching both the GeoDataFrame we take as the starting point and the value for `how` means that we in both cases take the tree data set as the starting point, but in the first case GeoPandas uses the spatial index of the larger dataset, while in the second example the spatial index of the smaller one is used.
**In this case the first version is ~10 times faster.**

Both of the latter exampels are using `intersect` as the spatial operation, so it is not as such changing the spatial operation that gives the performance boost, but rather the internal workings of which sindex is used.


## Efficient GeoPandas

It is generally a good idea to use the `sjoin` functions. 

Compare for example these methods for finding the nearest tree to each point in a small point data set:

**Slow method**

We only have a 100 points (but a lot of trees), but using a brute force method will still be *very* slow!

In [ ]:
points = gpd.read_parquet("files/points.parquet")

len(points)

In [ ]:
trees2 = gpd.read_parquet("files/trees.parquet")

In [ ]:
%%time
# NOTE do not run this cell unless you want to confirm that it indeed is slow
points.geometry.apply(lambda geom: trees2.distance(geom).min()) # the distances might be zero since both are from our very dense tree data set

**Slightly faster method:**

In [ ]:
from shapely.ops import nearest_points

In [ ]:
%%time

# unary union of the right geomtries (trees)
other_points = trees2.geometry.unary_union

def nearest_dist(point, other):
    
    # find the nearest point and return the distance to that point
    nearest = nearest_points(point, other)[1]
    return point.distance(nearest)

points.geometry.apply(lambda geom: nearest_dist(geom,other_points))

**Fast method:**

In [ ]:
%%time

points.sjoin_nearest(trees2[["geometry"]], how="left", distance_col="distance")

You can see more tips for writing efficient code with GeoPandas here: https://github.com/martinfleis/efficient-geopandas-workshop/blob/main/GeoPython2023.ipynb

## Dask-GeoPandas (to study at home)

<img src="files/dask.png" alt="Dask logo" width="150"/> <img src="files/gp.png" alt="Dask logo" width="112"/>

Many of the functionalities from GeoPandas are available in Dask-Geopandas. There are however some limitations in terms of functionalities in Dask-GeoPandas, and it is a new and sometimes slightly buggy libary.

GeoPandas can often be faster and easier to use than Dask-GeoPandas, so try GeoPandas first and turn to Dask if you have problems with performance, fitting data in memory, etc.

One common use case of Dask-GeoPandas is to read in large amounts of data, reduce it down, and then iterate on a much smaller amount of data. This may only be for a single component of your workflow, so it might make sense to revert back to GeoPandas once this component is complete.

**Useful links:**
* https://github.com/geopandas/dask-geopandas
* https://dask-geopandas.readthedocs.io/en/stable/


In [ ]:
import geopandas as gpd
import dask_geopandas
import numpy as np
import shapely.geometry
import matplotlib.pyplot as plt

If we have a regular GeoDataFrame, we can partition it into a Dask-GeoPandas DataFrame, specifying the number of partitions:

(see this [guide](https://blog.dask.org/2021/11/02/choosing-dask-chunk-sizes) for choosing the right number of partitions)

In [ ]:
gdf = gpd.read_file(gpd.datasets.get_path("naturalearth_lowres"))
gdf.to_crs("EPSG:8857",inplace=True)

gdf.plot();

In [ ]:
# Convert to dask-geopandas dataframe

dask_gdf = dask_geopandas.from_geopandas(gdf, npartitions=4)

If your dataset is too large to read into a regular GeoDataFrame efficiently, Dask-GeoPandas also supports [reading directly](https://dask-geopandas.readthedocs.io/en/stable/docs/reference/api/dask_geopandas.read_file.html) from a file.

Having a look at our `dask_gdf`, we can see the number of partitions (4), data types of all columns, and how many objects we have in each partition:

In [ ]:
dask_gdf

When we want to do a computation on a pandas/geopandas column, we simple run for example:

In [ ]:
gdf.geometry.area

For Dask-GeoPandas, we need to add `compute()`:

In [ ]:
# This does not actually compute the area (returns an empty result, since we haven't loaded the whole dataframe into memory)
dask_gdf.geometry.area

In [ ]:
dask_gdf.geometry.area.compute()

### Comparing performances with point-in-polygon

To see the benefits of using Dask-GeoPandas, let's do a quick point-in-polygon computation using random points:

In [ ]:
N = 10_000_000 # You can decrease this number if it takes too long to run on your machine

points = gpd.GeoDataFrame(geometry=gpd.points_from_xy(np.random.randn(N),np.random.randn(N)))

In [ ]:
# Dask version
dpoints =  dask_geopandas.from_geopandas(points, npartitions=16)

A single polygon for which we will check if the points are located within this polygon:

In [ ]:
box = shapely.geometry.box(0, 0, 1, 1)

**Let's first try to find the points within the box using the regular GeoDataFrame:**

In [ ]:
%%time
points.within(box)

**...and then with the Dask-GeoPandas dataframe:**

In [ ]:
%%time

dpoints.within(box).compute()


In this case, using dask-geopandas is much faster.

### Speeding up spatial operations

To compare the computational time, let's first turn our area and tree data from earlier into Dask-GeoPandas dataframes:

In [ ]:
dareas = dask_geopandas.from_geopandas(areas, npartitions=4)
dareas

In [ ]:
dtrees = dask_geopandas.from_geopandas(trees, npartitions=6)
dtrees

The "slow" type of spatial join from before can be done much faster with Dask - but Dask-Geopandas only supports "inner" joins at the moment...

**Using normal GeoDataFrames:**

In [ ]:
%%time
trees.sjoin(areas[['geometry','ValgstedId']], predicate="intersects", how="inner")

**Using Dask-GeoDataframes for both datasets:** 

(It might not save you CPU time, but the results will be ready sooner)

In [ ]:
%%time
dtrees.sjoin(dareas[['geometry','ValgstedId']], predicate="intersects", how="inner").compute()

### Spatial partitioning

So far we have just used the regular partitioning offered by Dask-GeoPandas, which, from a spatial standpoint, is a random partitioning just splitting the dataframe by rows.

Dask-GeoPandas uses `spatial_shuffle` to partition the data based on the geographical dimension.

To see this in practice, we will use a dataset of the Contiguous United States.

In [ ]:
usa = gpd.read_file("files/us_cont.gpkg")

In [ ]:
usa.plot(facecolor="none", linewidth=0.5, edgecolor="red");

Turn the GeoDataFrame into a Dask-GeoPandas dataframe (using regular partitioning):

In [ ]:
d_usa = dask_geopandas.from_geopandas(usa, npartitions=4) #
d_usa

By visualising the convex hull of each partition, we can get a feel for how the Dask-GeoDataFrame has been partitioned. A useful spatial partitioning scheme is one that minimises the degree of spatial overlap between partitions. By default, the standard partitions does a poor job of spatially partitioning our example data - there is a high degree of overlap between partitions.

In [ ]:
d_usa.calculate_spatial_partitions() # convex hull
d_usa.spatial_partitions

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(12,6))
usa.plot(ax=ax)
d_usa.spatial_partitions.plot(ax=ax, cmap="tab20", alpha=0.5)
ax.set_axis_off()
plt.show()

We can reduce the spatially overlapping partitions by using `spatial_shuffle()`.

Dask-GeoPandas supports [3 different methods](https://dask-geopandas.readthedocs.io/en/stable/guide/spatial-partitioning.html):

In [ ]:
hilbert = d_usa.spatial_shuffle(by="hilbert")
morton = d_usa.spatial_shuffle(by="morton")
geohash = d_usa.spatial_shuffle(by="geohash")

None of them completely removes the spatial overlap, but they all reduce it:

In [ ]:
fig, axes = plt.subplots(nrows=1,ncols=3, figsize=(25,12))
ax1, ax2, ax3 = axes.flatten()

for ax in axes:
    usa.plot(ax=ax)

hilbert.spatial_partitions.plot(ax=ax1, cmap="tab20", alpha=0.5)
morton.spatial_partitions.plot(ax=ax2, cmap="tab20", alpha=0.5)
geohash.spatial_partitions.plot(ax=ax3, cmap="tab20", alpha=0.5)

[axi.set_axis_off() for axi in axes.ravel()]

ax1.set_title("Hilbert", size=16)
ax2.set_title("Morton", size=16)
ax3.set_title("Geohash", size=16)

plt.show()

**Changing the number of partitions**

We can change the number of partitions when doing the spatial shuffling. If no number of partitions is specified when using `spatial_shuffle`, the outcome will have as many partitions as the original Dask-GeoPandas dataframe.

The ideal number of partitions depends on your data structure and how you plan to use your data.

In [ ]:
hilbert20 = d_usa.spatial_shuffle(by="hilbert", npartitions=20)

In [ ]:
fig, axes = plt.subplots(nrows=1,ncols=3, figsize=(25,12))
ax1, ax2, ax3 = axes.flatten()

for ax in axes:
    usa.plot(ax=ax)

d_usa.spatial_partitions.plot(ax=ax1, cmap="tab20", alpha=0.5)
hilbert.spatial_partitions.plot(ax=ax2, cmap="tab20", alpha=0.5)
hilbert20.spatial_partitions.plot(ax=ax3, cmap="tab20", alpha=0.5)

[axi.set_axis_off() for axi in axes.ravel()]

ax1.set_title("No spatial shuffle, with 4 partitions", size=16)
ax2.set_title("Spatial shuffle using default npartitions", size=16)
ax3.set_title("Spatial shuffle using 20 partitions", size=16)

plt.show()

**Why spatial partitioning?**

If you are querying for a small spatial subset of your data, using spatial partitioning means than we can ignore the data in the other partitions. At the moment, it is still an [open issue](https://notebooksharing.space/view/88055f29ae1c26b22f61a1ef5f673cf971f434f2e513933d8de2001d7f49162a#displayOptions=) to make Dask-GeoPandas default utilizing spatial partitioning to improve spatial operations. Spatial partitioning can however be used to only load the data into memory that you need - see below.

### Exporting Dask-Geopandas

Dask-GeoPandas only supports writing files to `feather` and `parquet`. This will export each partition to its own file.

In [ ]:
dask_gdf.to_parquet("files/dask_export.parquet")

In [ ]:
ddf = dask_geopandas.read_parquet("files/dask_export.parquet", gather_spatial_partitions=False)
ddf.spatial_partitions  # None

**Exporting and reading a spatially partitioned dataset:**

Whenever we save a partitioned dataset we have the option of only reading part of the partitions. If we have used spatial partitioning we can make use of this to only include data for the area we are interested in.

In [ ]:
# Save spatially partitioned data
geohash.to_parquet("files/geohash.parquet")

In [ ]:
# read only the first partition
part0 = dask_geopandas.read_parquet("files/geohash.parquet/part.0.parquet")

In [ ]:
# Plot data to see what the first partition includes
part0.head(len(part0)).plot();

If we don't know beforehand which spatial partition we are interested in, we can read the whole file and query the spatial partitioning.
Dask-GeoPandas can return the extent of the spatial partitions without reading the entire dataset into memory. Once you know which partition you are interested in, you can limit the rest of the analysis to that partition alone.

Imagine you have a *very* big dataset with US data and only want to load the partition that contains the location of New York:

* First, we 'read' the whole parquet file. This loads the meta data without reading the entire file into memory:

In [ ]:
geohash_from_parquet = dask_geopandas.read_parquet("files/geohash.parquet/")

geohash_from_parquet.spatial_partitions

* Then, we use the polygons for the spatial partitions to check which contain the location of NY. Note that the partitions can be overlapping, so it might be more than one:

In [ ]:
from shapely.geometry import Point
ny_coords = Point(-74.0059413,40.7127837)

In [ ]:
for i, polygon in geohash_from_parquet.spatial_partitions.items():
   if polygon.intersects(ny_coords):
      print(f"New York is in partition {i}")

Now we know that we only have to work with partition 3 (this small example only uses a point to represent the location of New York. To get an accurate result, we would of course need to use a polygon showing the true extent of the city).


## H3

<img src="files/h3.png" alt="H3 logo" width="150"/>

H3 is both a useful tool for spatial indexing, aggregation, and spatial queries and supports some type of analysis, such as surface interpolation, clustering etc.

**Useful links:**

* [Official H3 website](https://h3geo.org/)
* [h3-py](https://uber.github.io/h3-py/intro.html)
* [Example notebooks](https://github.com/uber/h3-py-notebooks)
* A few other examples of using H3 in geospatial analysis:
    * https://towardsdatascience.com/geographic-clustering-with-hdbscan-ef8cb0ed6051
    * https://towardsdatascience.com/uber-h3-for-data-analysis-with-python-1e54acdcc908
    * https://betterprogramming.pub/playing-with-ubers-hexagonal-hierarchical-spatial-index-h3-ed8d5cd7739d 


We use the H3-py library from Uber here, but see also [H3-Pandas](https://h3-pandas.readthedocs.io/en/latest/index.html).

In [ ]:
import h3
import matplotlib.pyplot as plt
import rasterio
import geopandas as gpd
import pandas as pd
from shapely.geometry import Polygon
import matplotlib.pyplot as plt
from rasterio.plot import show
import rioxarray as rxr
import contextily as cx 

### Vectorizing raster data with H3

One great use case of H3 hexagons is fast conversion of raster/image data to vector format.
In this example we will use a raster with population densities downloaded from the [Global Human Settlement](https://ghsl.jrc.ec.europa.eu/download.php?ds=pop) database.

The raster has been preprocessed to make sure that we have one coherent raster for all of Denmark, and that it is in a CRS compatible with H3 (WGS84).

In [ ]:
# Small helper function for plotting

def plot_scatter(df, metric_col, x='lng', y='lat', marker='.', alpha=1, figsize=(16,12), colormap='viridis'):   

    '''
    Helper function from H3 tutorials: https://github.com/uber/h3-py-notebooks
    ''' 
    df.plot.scatter(x=x, y=y, c=metric_col, title=metric_col
                    , edgecolors='none', colormap=colormap, marker=marker, alpha=alpha, figsize=figsize);
    plt.xticks([], []); plt.yticks([], [])

In [ ]:
# load pop raster and plot
pop_raster = rasterio.open("files/reproj_pop_raster_wgs84.tif")

show(pop_raster);

First, we use a xarray meethod `to_pandas()` to convert our raster data to a vector point dataset. This is called *vectorization*.

In [ ]:
# Convert raster to pandas dataframe with coordinates and pop values

pop_df = (
    rxr.open_rasterio("files/reproj_pop_raster_wgs84.tif")
    .sel(band=1)
    .to_pandas()
    .stack()
    .reset_index()
    .rename(columns={"x": "lng", "y": "lat", 0: "population"})
)

pop_df

A bit of processing of our pandas data + turn it into a GeoDataFrame:

In [ ]:
# Ignore no data values
pop_df = pop_df[pop_df.population > -200]

# Convert to GeoDataFrame
pop_gdf = gpd.GeoDataFrame(pop_df, geometry=gpd.points_from_xy(pop_df.lng, pop_df.lat))

# Define CRS
pop_gdf.set_crs("EPSG:4326", inplace=True)


As we saw when we plotted the raster, we also have some data for Sweden. To make sure our final dataset only covers Denmark, let's use our `areas` data to cut the population data to the right extent.

In [ ]:
# Dissolve to one big geometry and project to same crs as pop data

dk_gdf = areas.dissolve().to_crs("EPSG:4326")

dk_gdf.plot();

Doing a spatial join with `sjoin` and `predicate="inner"` gives us a GeoDataFrame only with geometries where the two input datasets overlap:

In [ ]:
# Inner spatial join of pop points and DK boundary

joined_gdf = gpd.sjoin(pop_gdf, dk_gdf, predicate="within", how="inner")

plot_scatter(joined_gdf, metric_col="population", marker=".", colormap="viridis")

We have now turned our population raster into a population point dataset. However, if we want to combine our population data with other layers, you will often want to aggregate - for example with H3.

You can see the average sizes of the H3 hexagons [here](https://h3geo.org/docs/core-library/restable/).

Below we use the H3 method `latlng_to_cell` to return the H3 index based on a point coordinate and a specificed H3 resolution.
We also use `cell_to_boundary` which returns the hex polygon associated with a hex id as GeoJSON.

In [ ]:
# INDEX POPULATION AT VARIOUS H3 LEVELS
for res in range(6, 9):
    col_hex_id = "hex_id_{}".format(res)
    col_geom = "geometry_{}".format(res)
    msg_ = "At resolution {} -->  H3 cell id : {}"
    print(msg_.format(res, col_hex_id, col_geom))

    joined_gdf[col_hex_id] = joined_gdf.apply(
        lambda row: h3.latlng_to_cell(lat=row["lat"], lng=row["lng"], res=res), axis=1
    )
   

Now we have the id of the H3 hexagons for each data point at resolution 6 to 8:

In [ ]:
joined_gdf.head()

There are however many data points in each hex, so get get a correct aggregation, we need to summarize the number of people in each hex cell:

In [ ]:
hex_id_col = "hex_id_7"
h3_groups = (
    joined_gdf.groupby(hex_id_col)["population"].sum().to_frame("population").reset_index()
)

# Get the coordinate of the centroid of the hexagons
h3_groups["lat"] = h3_groups[hex_id_col].apply(lambda x: h3.cell_to_latlng(x)[0])
h3_groups["lng"] = h3_groups[hex_id_col].apply(lambda x: h3.cell_to_latlng(x)[1])

# use h3.h3_to_geo_boundary to obtain the geometries of these hexagons
h3_groups["hex_geometry"] = h3_groups[hex_id_col].apply(
    lambda x: {
        "type": "Polygon",
        "coordinates": [h3.cell_to_boundary(h=x)],
    }
)

In [ ]:
h3_groups.head()

The hex geometries are still just stored as GeoJSON in a column, while the geometry column uses the centroid of the hex grids at our chosen resolution. If we plot we can see that it is just the centroids:

In [ ]:
h3_groups.plot.scatter(
    x="lng",
    y="lat",
    c="population",
    marker="o",
    edgecolors="none",
    colormap="viridis",
    figsize=(30, 20),
)
plt.xticks([], [])
plt.yticks([], [])
plt.title("hex-grid: population");

Therefore, the final step is to use the H3 hexagon as the geometries. Note how we have to use `h3.LatLngPoly` instead of `Polygon` due to switched orders of latitudes and longitudes - a common pitfall!

In [ ]:
h3_groups["geometry"] = h3_groups["hex_geometry"].apply(
    lambda x: h3.LatLngPoly(list(x["coordinates"][0]))
)

h3_gdf = gpd.GeoDataFrame(h3_groups, geometry="geometry", crs="EPSG:4326")


...and to see the final results:

In [ ]:
fig, ax = plt.subplots(figsize=(15,15))

ax.set_facecolor("black")

# Turn ax ticks off without affecting face color
for spine in ax.spines.values():
    spine.set_visible(False)
ax.tick_params(bottom=False, labelbottom=False,
               left=False, labelleft=False)

h3_gdf.plot(ax=ax,column='population', scheme='fisherjenks', cmap='Blues', legend=True)

plt.title("DK Population",fontsize=18);

### Data aggregation of global shipping data

We can use a similar approach to aggregating this global data set on maritime piracy:

(Data from [Maritime Safety Information](https://msi.nga.mil/NGAPortal/MSI.porta), tutorial from [Spatial Thoughts](https://spatialthoughts.com/2020/07/01/point-in-polygon-h3-geopandas/)).

In [ ]:
zipfile = 'zip://files/ASAM_shp.zip/asam_data_download/ASAM_events.shp'
incidents = gpd.read_file(zipfile)
incidents.head()

This time, we use resolution 3 (so much bigger hexagons) for the data aggregation.

A small helper function helps us get the hex grid id corresponding to our point geometries: Notice the new column 'h3' after running the following cell:

In [ ]:
h3_level = 3

def lat_lng_to_h3(row):
    return h3.latlng_to_cell(row.geometry.y, row.geometry.x, h3_level)

incidents['h3'] = incidents.apply(lat_lng_to_h3, axis=1)
incidents.head()

In the population example we took the sum of all population points in each hex grid. In this case each point represents one incident, so we use count to aggregate:

In [ ]:
counts = incidents.groupby(['h3']).h3.agg('count').to_frame('count').reset_index()
counts.sample(5)

Just like in the first example, we want to get the polygons corresponding to the hex grid ids:

In [ ]:
def add_geometry(row):
    points = h3.cell_to_boundary(row['h3'])
    return h3.LatLngPoly(points)

counts['geometry'] = counts.apply(add_geometry, axis=1)
counts.head()

Finally, we turn it into a GeoDataFrame and specify the CRS:

In [ ]:
gdf = gpd.GeoDataFrame(counts, crs='EPSG:4326')

In [ ]:
fig, ax = plt.subplots(figsize=(15,15))

# ax.set_axis_off() # Does not seem to work anymore

ax.set_facecolor("black")

# Turn ax ticks off without affecting face color
for spine in ax.spines.values():
    spine.set_visible(False)
ax.tick_params(bottom=False, labelbottom=False,
               left=False, labelleft=False)

gdf.plot(ax=ax,column='count', scheme='fisherjenks',  cmap='Reds', legend=True)
cx.add_basemap(ax=ax, crs=gdf.crs, source=cx.providers.CartoDB.DarkMatterNoLabels) # Does not seem to work anymore

plt.title("Maritime Piracy",fontsize=18);